# Drone RL walkthrough (runnable)

The headless-safe parts of [`TUTORIAL.md`](TUTORIAL.md), cell by cell.

**Kernel:** the `drones` conda environment. Start Jupyter from that env:

```bash
conda activate drones
pip install jupyter        # once
cd path/to/CMS/tutorial
jupyter lab walkthrough.ipynb
```

Environment setup (installing conda, the C++ compiler, `pip install -e .`) is in
`TUTORIAL.md` steps 2-3 and can't run from inside a notebook.

## 1. Check the install

In [ ]:
import torch, stable_baselines3, gymnasium, pybullet
import gym_pybullet_drones

print('torch              ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('stable-baselines3  ', stable_baselines3.__version__)
print('gymnasium          ', gymnasium.__version__)
print('gym-pybullet-drones', gym_pybullet_drones.__version__)
# cuda: False is expected on a laptop with no NVIDIA GPU - training runs on CPU.

## 2. Inspect the environment

`HoverAviary`: one Crazyflie, hold `(0, 0, 1.0)` m for 8 s.

In [ ]:
import numpy as np
from gym_pybullet_drones.envs.HoverAviary import HoverAviary
from gym_pybullet_drones.utils.enums import ObservationType, ActionType

env = HoverAviary(obs=ObservationType.KIN, act=ActionType.ONE_D_RPM)
obs, _ = env.reset(seed=0)

print('observation space:', env.observation_space)   # (1, 27): pose + velocity + action history
print('action space:     ', env.action_space)        # (1, 1): one thrust scalar for all 4 motors

obs, reward, term, trunc, info = env.step(np.array([[0.0]], dtype=np.float32))
print('reward after one step:', float(reward))
env.close()

## 3. A short training run

20k steps just to see the loop (~1-2 min on CPU). The real run is
`python ../scripts/quickstart_train.py` (300k steps, early-stops ~160k, ~7 min).

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

OBS, ACT = ObservationType.KIN, ActionType.ONE_D_RPM
train_env = make_vec_env(HoverAviary, env_kwargs=dict(obs=OBS, act=ACT), n_envs=4, seed=0)

model = PPO('MlpPolicy', train_env, verbose=0)
model.learn(total_timesteps=20_000)
print('done - trained', model.num_timesteps, 'steps')
train_env.close()

## 4. Evaluate the *fully* trained policy

Uses the checkpoint committed at `../results/best_model.zip` (474/474).

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

best = PPO.load('../results/best_model.zip')
eval_env = HoverAviary(obs=OBS, act=ACT)
mean_r, std_r = evaluate_policy(best, eval_env, n_eval_episodes=10)
print(f'mean reward: {mean_r:.1f} +/- {std_r:.1f}   (474 ~ perfect hover)')
eval_env.close()

In [ ]:
import matplotlib.pyplot as plt

env = HoverAviary(obs=OBS, act=ACT)
obs, _ = env.reset(seed=123)
zs, drift, ts = [], [], []
for i in range((env.EPISODE_LEN_SEC + 2) * env.CTRL_FREQ):
    action, _ = best.predict(obs, deterministic=True)
    obs, _, term, trunc, _ = env.step(action)
    s = obs.squeeze()
    zs.append(s[2]); drift.append(np.hypot(s[0], s[1])); ts.append(i / env.CTRL_FREQ)
    if term or trunc:
        break
env.close()

fig, ax = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
ax[0].axhline(1.0, ls='--', c='gray'); ax[0].plot(ts, zs)
ax[0].set_ylabel('altitude z (m)'); ax[0].grid(alpha=.3)
ax[1].plot(ts, drift, c='C3'); ax[1].set_ylabel('drift (m)'); ax[1].set_xlabel('time (s)')
ax[1].grid(alpha=.3); fig.tight_layout(); plt.show()

## 5. Build the basketball court (headless render)

`court.py` lives in `../scripts`. This builds the court in a `DIRECT` (windowless)
PyBullet client and grabs a camera image. The live version is
`python ../scripts/fly_in_court.py`.

In [ ]:
import sys, pybullet as p, pybullet_data
sys.path.insert(0, '../scripts')
import court

c = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
info = court.build_court(c, scale=0.5)

# a marker where the drone would hover
vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.09, 0.09, 0.02], rgbaColor=[0.1, 0.1, 0.1, 1])
p.createMultiBody(0, -1, vis, [1.5, 0.6, 1.0])

w, h = 960, 540
view = p.computeViewMatrixFromYawPitchRoll([0, 0, 0.5], info['length'] * 0.8, 55, -40, 0, 2)
proj = p.computeProjectionMatrixFOV(55, w / h, 0.1, 100)
rgb = np.reshape(p.getCameraImage(w, h, view, proj)[2], (h, w, 4))[:, :, :3]
p.disconnect(c)

print('court:', info['length'], 'x', info['width'], 'm   rim height', round(info['rim_height'], 2), 'm')
plt.figure(figsize=(9, 5)); plt.imshow(rgb); plt.axis('off'); plt.show()
print('(PyBullet\'s live GUI renders the texture much crisper than this headless capture.)')

## 6. Next

- Fly the PID waypoint tour of the court (opens a window):
  `python ../scripts/fly_in_court.py`  (add `--record` for an mp4)
- Drop the RL policy into the court: `python ../scripts/fly_in_court.py --rl`
  (hovers at centre only - it was trained with the 1-D thrust action space)
- Full next-steps list: [`../PROGRESS.md`](../PROGRESS.md)